# 그룹 · 집계 · 인덱스

작업형 1유형 연습 — 포함: T1-6, T1-12, T1-14

> 데이터는 seaborn/sklearn 내장셋(titanic, tips, penguins 등)으로 대체해 연습.
> 문제 지문은 원문 복붙 대신 **본인 말로 요약**해서 적기 (출처: dataq 체험환경 / 캐글 등).

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns

In [3]:
df = sns.load_dataset('titanic')

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 15 columns):
 #   Column       Non-Null Count  Dtype   
---  ------       --------------  -----   
 0   survived     891 non-null    int64   
 1   pclass       891 non-null    int64   
 2   sex          891 non-null    object  
 3   age          714 non-null    float64 
 4   sibsp        891 non-null    int64   
 5   parch        891 non-null    int64   
 6   fare         891 non-null    float64 
 7   embarked     889 non-null    object  
 8   class        891 non-null    category
 9   who          891 non-null    object  
 10  adult_male   891 non-null    bool    
 11  deck         203 non-null    category
 12  embark_town  889 non-null    object  
 13  alive        891 non-null    object  
 14  alone        891 non-null    bool    
dtypes: bool(2), category(2), float64(2), int64(4), object(5)
memory usage: 80.7+ KB


value_counts()는 원래 1차원(Series) 전용 함수였지만, 판다스 버전이 업데이트되면서 2차원(DataFrame) 데이터셋도 지원하도록 기능이 확장됨 - 그래서 여러개도 동시에 가능함.

In [9]:
print(df['survived'].value_counts())

print(df[['survived', 'sex']].value_counts())


survived
0    549
1    342
Name: count, dtype: int64
survived  sex   
0         male      468
1         female    233
          male      109
0         female     81
Name: count, dtype: int64


## Groupby 합계

- 타이타닉 데이터셋에서 객실 등급(pclass)과 성별(sex)에 따른 생존자(survived) 수의 합계를 구하시오.

- **풀이 메모:** groupby()를 적용, .sum() 활용하기

In [6]:

groupby_sum = df.groupby(['pclass', 'sex'])['survived'].sum()
print(groupby_sum)

pclass  sex   
1       female    91
        male      45
2       female    70
        male      17
3       female    72
        male      47
Name: survived, dtype: int64


## 상하위 10개 추출

**문제 :** 타이타닉 데이터셋에서 탑승객 요금(fare)을 기준으로 상위 10개 데이터와 하위 10개 데이터를 추출하여 각각의 요금 평균을 구하고, 그 차이(절댓값)를 산출

**풀이 메모:** sort_values()로 정렬, .head(), .tail() 사용

In [12]:
df_sorted = df.sort_values(by='fare', ascending=True)

top10_mean = df_sorted.head(10)['fare'].mean()
tail10_mean = df_sorted.tail(10)['fare'].mean()

print(abs(top10_mean - tail10_mean))


336.12584000000004


## MultiIndex + Groupby

문제: 타이타닉 데이터셋에서 객실 등급(pclass)과 성별(sex)로 그룹을 나누어 요금(fare)의 평균을 구한 뒤, '1등급 여성'의 평균 요금과 '3등급 남성'의 평균 요금의 차이(절댓값)를 반환하시오. (소수점 아래셋째 자리에서 반올림하여 둘째 자리까지 표시)

풀이 메모: groupby() 결과로 생성된 멀티인덱스 시리즈에서 .loc[]를 사용해 특정 인덱스 조합의 값을 추출하고 계산한다. 

In [14]:

groupby_mean = df.groupby(['pclass', 'sex'])['fare'].mean()


groupby_mean

pclass  sex   
1       female    106.125798
        male       67.226127
2       female     21.970121
        male       19.741782
3       female     16.118810
        male       12.661633
Name: fare, dtype: float64

### 왜 loc을 쓰는가??

.loc를 써야 하는 이유는 내가 원하는 특정 행(Row)을 정확하게 짚어서 꺼내오기 위한 판다스의 표준 문법이기 때문.

위 코드를 실행하면 아래와 같은 구조의 멀티인덱스 시리즈가 만들어집니다. 테이블처럼 보이지만, 왼쪽의 pclass와 sex는 컬럼이 아니라 이중 구조로 겹쳐진 '인덱스(주소)'. 

컬럼을 고르는 게 아니라, '1등급(pclass=1)이면서 여성(sex=female)인 행(Row)'을 찾아가서 그 값을 뽑아내는 작업일 때는, groupby_mean.loc[(1, 'female')] (O): .loc는 "인덱스(행 이름)를 기준으로 찾겠다"고 명시하도록 써야 함!


In [17]:
# 1등급 + 여성 요금
p1_female = groupby_mean.loc[(1, 'female')]
print(p1_female)

# 3등급 남성의 평균 요금
p3_male = groupby_mean.loc[(3, 'male')]
print(p3_male)


# 차이 계산
result = round(abs(p1_female - p3_male), 2)

print(result)


106.12579787234043
12.661632564841499
93.46
